# 임도망 기반 산불 진화 접근성 EDA

임도는 산림 작업 도로로, 산불이 발생했을 때 진화 인력과 장비가 얼마나 빨리 접근할 수 있는지를 판단하는 중요한 변수입니다.

분석 목표:
- 산불 발생 지점과 가장 가까운 임도 사이의 거리 계산
- 임도와 가까운 산불/먼 산불 분포 비교
- 임도 접근성이 낮아 보이는 지역 격자 찾기
- 임도 길이와 행정구역별 분포 확인

주의: 이 노트북은 별도 GIS 패키지 없이 실행되도록 만들었습니다. 임도 `LINESTRING`의 좌표점을 일정 간격으로 샘플링해 최근접 거리를 근사 계산하므로, 정밀 GIS 분석에서는 `geopandas`, `shapely`, `pyproj` 기반 선분 거리 계산으로 고도화할 수 있습니다.

In [ ]:
from pathlib import Path
import math
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.rcParams["font.family"] = ["Malgun Gothic", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid", font="Malgun Gothic")

ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
ROAD_PATH = ROOT / "data" / "processed" / "전국_임도망도.csv"
FIRE_PATH = ROOT / "data" / "processed" / "산불발생위치도_지형특성계산.csv"
OUT_DIR = ROOT / "outputs" / "forest_road_eda"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ROAD_PATH, FIRE_PATH

## 1. 데이터 로드

In [ ]:
roads = pd.read_csv(ROAD_PATH)
fires = pd.read_csv(FIRE_PATH)

print("임도 데이터:", roads.shape)
print("산불 데이터:", fires.shape)
display(roads.head(3))
display(fires.head(3))

In [ ]:
roads.info()

## 2. 임도 기본 EDA

행정구역별 임도 개수와 총 길이를 확인합니다. `도형길이`는 미터 단위로 보고 km로 변환합니다.

In [ ]:
roads["도형길이_km"] = roads["도형길이"] / 1000

admin_summary = (
    roads.groupby("행정구역")
    .agg(
        임도수=("임도명", "count"),
        총임도길이_km=("도형길이_km", "sum"),
        평균폭_m=("폭(m)", "mean"),
        평균임도시설거리=("임도시설거리", "mean"),
    )
    .sort_values("총임도길이_km", ascending=False)
)

admin_summary.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

top_len = admin_summary.head(12).reset_index()
sns.barplot(data=top_len, y="행정구역", x="총임도길이_km", ax=axes[0], color="#22c55e")
axes[0].set_title("행정구역별 총 임도 길이 Top 12")
axes[0].set_xlabel("총 임도 길이(km)")
axes[0].set_ylabel("")

sns.histplot(roads["도형길이_km"].dropna(), bins=40, kde=True, ax=axes[1], color="#3b82f6")
axes[1].set_title("개별 임도 길이 분포")
axes[1].set_xlabel("임도 길이(km)")
axes[1].set_ylabel("임도 수")

plt.tight_layout()
plt.savefig(OUT_DIR / "road_basic_eda.png", dpi=160)
plt.show()

## 3. 임도 좌표 파싱 및 지도 확인

`공간좌표`의 `LINESTRING (경도 위도, ...)`에서 좌표를 추출합니다. 접근성 거리 계산 속도를 위해 임도 좌표점은 일정 간격으로 샘플링합니다.

In [ ]:
coord_pattern = re.compile(r"([0-9]+\.[0-9]+) ([0-9]+\.[0-9]+)")

def parse_linestring_points(wkt, step=10):
    points = coord_pattern.findall(str(wkt))
    if len(points) > step:
        points = points[::step]
    return [(float(lon), float(lat)) for lon, lat in points]

road_lons = []
road_lats = []

for wkt in roads["공간좌표"].dropna():
    for lon, lat in parse_linestring_points(wkt, step=10):
        road_lons.append(lon)
        road_lats.append(lat)

road_points = pd.DataFrame({"경도": road_lons, "위도": road_lats})
print("샘플링된 임도 좌표점 수:", len(road_points))
road_points.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 9))
ax.scatter(road_points["경도"], road_points["위도"], s=0.3, c="#64748b", alpha=0.25, label="임도")
ax.scatter(fires["경도"], fires["위도"], s=1.5, c="#ef4444", alpha=0.25, label="산불")
ax.set_title("전국 임도망과 산불 발생 지점")
ax.set_xlabel("경도")
ax.set_ylabel("위도")
ax.legend(markerscale=8)
plt.tight_layout()
plt.savefig(OUT_DIR / "road_fire_map.png", dpi=180)
plt.show()

## 4. 산불 지점과 가장 가까운 임도 거리 계산

위경도를 미터 단위 평면 좌표로 근사 변환한 뒤, 10km 격자 인덱스를 만들어 가까운 임도 후보만 비교합니다.

거리 구간 해석 예시:
- 1km 이내: 임도 접근성이 매우 좋은 산불 지점
- 1~5km: 비교적 접근 가능
- 5~10km: 접근 시간이 길어질 수 있음
- 10km 초과: 초기 진화 접근성이 낮을 가능성이 큼

In [ ]:
def lonlat_to_meter(lon, lat, lat0=36.5):
    radius = 6_371_000
    x = np.deg2rad(lon) * radius * np.cos(np.deg2rad(lat0))
    y = np.deg2rad(lat) * radius
    return x, y

road_x, road_y = lonlat_to_meter(road_points["경도"].to_numpy(), road_points["위도"].to_numpy())
fire_x, fire_y = lonlat_to_meter(fires["경도"].to_numpy(), fires["위도"].to_numpy())

cell_size = 10_000
road_ix = np.floor(road_x / cell_size).astype(int)
road_iy = np.floor(road_y / cell_size).astype(int)

grid = {}
for idx, key in enumerate(zip(road_ix, road_iy)):
    grid.setdefault(key, []).append(idx)

def nearest_road_distances(fx, fy, max_radius_cells=25):
    nearest = np.empty(len(fx), dtype=float)

    for i, (x, y) in enumerate(zip(fx, fy)):
        cx = int(math.floor(x / cell_size))
        cy = int(math.floor(y / cell_size))
        candidates = []

        for radius in range(max_radius_cells + 1):
            for gx in range(cx - radius, cx + radius + 1):
                for gy in range(cy - radius, cy + radius + 1):
                    if radius == 0 or gx in (cx - radius, cx + radius) or gy in (cy - radius, cy + radius):
                        candidates.extend(grid.get((gx, gy), []))

            if candidates:
                dx = road_x[candidates] - x
                dy = road_y[candidates] - y
                nearest[i] = np.sqrt(dx * dx + dy * dy).min()
                break
        else:
            nearest[i] = np.nan

    return nearest

fires_access = fires.copy()
fires_access["nearest_road_m"] = nearest_road_distances(fire_x, fire_y)
fires_access["nearest_road_km"] = fires_access["nearest_road_m"] / 1000

bins = [0, 1, 5, 10, 20, np.inf]
labels = ["1km 이내", "1~5km", "5~10km", "10~20km", "20km 초과"]
fires_access["임도거리구간"] = pd.cut(fires_access["nearest_road_km"], bins=bins, labels=labels, right=False)

fires_access[["fire_id", "위도", "경도", "nearest_road_km", "임도거리구간"]].head()

In [ ]:
distance_summary = fires_access["nearest_road_km"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95])
distance_bins = (
    fires_access["임도거리구간"]
    .value_counts(dropna=False)
    .reindex(labels)
    .rename_axis("임도거리구간")
    .reset_index(name="산불건수")
)
distance_bins["비율_%"] = distance_bins["산불건수"] / distance_bins["산불건수"].sum() * 100

display(distance_summary.round(2))
display(distance_bins.round(2))

fires_access.to_csv(OUT_DIR / "fire_nearest_forest_road_distance.csv", index=False, encoding="utf-8-sig")
distance_bins.to_csv(OUT_DIR / "fire_road_distance_bins.csv", index=False, encoding="utf-8-sig")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

sns.histplot(fires_access["nearest_road_km"].dropna(), bins=50, kde=True, ax=axes[0], color="#ef4444")
axes[0].axvline(5, color="#334155", linestyle="--", linewidth=1)
axes[0].axvline(10, color="#334155", linestyle="--", linewidth=1)
axes[0].set_title("산불 지점-최근접 임도 거리 분포")
axes[0].set_xlabel("최근접 임도 거리(km)")
axes[0].set_ylabel("산불 발생 건수")

sns.barplot(data=distance_bins, x="임도거리구간", y="산불건수", ax=axes[1], color="#f97316")
axes[1].set_title("임도 거리 구간별 산불 발생 건수")
axes[1].set_xlabel("최근접 임도 거리 구간")
axes[1].set_ylabel("산불 발생 건수")
axes[1].tick_params(axis="x", rotation=20)

for i, row in distance_bins.iterrows():
    axes[1].text(i, row["산불건수"], f"{row['비율_%']:.1f}%", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / "fire_road_distance_distribution.png", dpi=160)
plt.show()

## 5. 임도와 먼 산불 지점 지도

최근접 임도 거리가 멀수록 초기 진화 접근성이 낮을 가능성이 있습니다.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 9))
ax.scatter(road_points["경도"], road_points["위도"], s=0.2, c="#94a3b8", alpha=0.18, label="임도")
sc = ax.scatter(
    fires_access["경도"],
    fires_access["위도"],
    c=fires_access["nearest_road_km"],
    s=3,
    cmap="magma_r",
    alpha=0.65,
    vmin=0,
    vmax=fires_access["nearest_road_km"].quantile(0.95),
)
plt.colorbar(sc, ax=ax, label="최근접 임도 거리(km)")
ax.set_title("산불 발생 지점별 임도 접근 거리")
ax.set_xlabel("경도")
ax.set_ylabel("위도")
plt.tight_layout()
plt.savefig(OUT_DIR / "fire_road_accessibility_map.png", dpi=180)
plt.show()

## 6. 임도 접근 취약 격자 찾기

산불 발생 지점을 0.5도 격자로 묶고, 각 격자의 평균/중앙 임도 거리와 산불 건수를 계산합니다. 산불 건수가 어느 정도 있으면서 최근접 임도 거리가 긴 격자는 진화 접근성 관점에서 우선 검토 대상입니다.

In [ ]:
grid_size_deg = 0.5
fires_access["lon_grid"] = np.floor(fires_access["경도"] / grid_size_deg) * grid_size_deg
fires_access["lat_grid"] = np.floor(fires_access["위도"] / grid_size_deg) * grid_size_deg

access_grid = (
    fires_access.groupby(["lat_grid", "lon_grid"])
    .agg(
        산불건수=("fire_id", "count"),
        평균임도거리_km=("nearest_road_km", "mean"),
        중앙임도거리_km=("nearest_road_km", "median"),
        최대임도거리_km=("nearest_road_km", "max"),
    )
    .reset_index()
)

vulnerable_grids = (
    access_grid[access_grid["산불건수"] >= 30]
    .sort_values(["평균임도거리_km", "산불건수"], ascending=[False, False])
    .head(15)
)

access_grid.to_csv(OUT_DIR / "forest_road_access_grid.csv", index=False, encoding="utf-8-sig")
vulnerable_grids.to_csv(OUT_DIR / "forest_road_vulnerable_grids.csv", index=False, encoding="utf-8-sig")

vulnerable_grids.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 9))
sc = ax.scatter(
    access_grid["lon_grid"] + grid_size_deg / 2,
    access_grid["lat_grid"] + grid_size_deg / 2,
    c=access_grid["평균임도거리_km"],
    s=np.clip(access_grid["산불건수"], 10, 300),
    cmap="YlOrRd",
    alpha=0.75,
)
plt.colorbar(sc, ax=ax, label="평균 최근접 임도 거리(km)")
ax.set_title("격자별 산불 발생 건수와 임도 접근 취약도")
ax.set_xlabel("경도")
ax.set_ylabel("위도")
plt.tight_layout()
plt.savefig(OUT_DIR / "forest_road_vulnerable_grid_map.png", dpi=180)
plt.show()

## 7. 인사이트 정리

In [ ]:
n = len(fires_access)
median_dist = fires_access["nearest_road_km"].median()
mean_dist = fires_access["nearest_road_km"].mean()
over_5 = (fires_access["nearest_road_km"] >= 5).sum()
over_10 = (fires_access["nearest_road_km"] >= 10).sum()
top_admin = admin_summary.iloc[0]
top_admin_name = admin_summary.index[0]

print(f"- 분석 대상 산불 발생 지점은 총 {n:,}건입니다.")
print(f"- 산불 지점과 최근접 임도 사이의 평균 거리는 {mean_dist:.1f}km, 중앙값은 {median_dist:.1f}km입니다.")
print(f"- 임도에서 5km 이상 떨어진 산불 지점은 {over_5:,}건({over_5 / n * 100:.1f}%)입니다.")
print(f"- 임도에서 10km 이상 떨어진 산불 지점은 {over_10:,}건({over_10 / n * 100:.1f}%)입니다.")
print(f"- 총 임도 길이가 가장 긴 행정구역은 {top_admin_name}({top_admin['총임도길이_km']:.1f}km)입니다.")
print("- 평균 임도 거리가 긴 격자는 초기 진화 접근성이 낮을 가능성이 있어 우선 검토 대상입니다.")

### 보고서에 쓸 수 있는 문장

- 임도와 가까운 산불 지점은 진화 인력과 장비의 접근성이 상대적으로 좋을 가능성이 있다.
- 임도에서 5km 또는 10km 이상 떨어진 산불 지점은 초기 진화 접근 시간이 길어질 수 있어 별도 관리가 필요하다.
- 산불 발생 빈도가 있으면서 평균 임도 거리가 긴 격자는 진화 접근 취약 지역 후보로 볼 수 있다.
- 단, 본 분석은 임도 좌표점 샘플링 기반 근사 거리이므로, 최종 정책 판단에는 실제 도로망, 등산로, 지형 장애물, 소방 출동 거점까지 함께 고려해야 한다.